In [5]:
!pip3 install torch torchvision albumentations opencv-python matplotlib


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


## 1. Imports + Config

In [10]:
import os
import cv2
import json
import numpy as np
from collections import Counter

# Config
TRAIN_DIR = "dataset/train"
TEST_DIR = "dataset/test"

os.makedirs("artifacts", exist_ok=True)

## 2. Load Data Function

In [11]:
def load_data(data_dir):
    image_paths = []
    labels = []

    # Sort classes for consistency
    classes = sorted(os.listdir(data_dir))
    class_to_idx = {cls: i for i, cls in enumerate(classes)}

    for cls in classes:
        cls_path = os.path.join(data_dir, cls)

        if not os.path.isdir(cls_path):
            continue

        for img_name in os.listdir(cls_path):
            if not img_name.lower().endswith(('.png', '.jpg', '.jpeg')):
                continue

            img_path = os.path.join(cls_path, img_name)

            # Optional: check image validity
            img = cv2.imread(img_path)
            if img is None:
                continue

            image_paths.append(img_path)
            labels.append(class_to_idx[cls])

    return image_paths, labels, class_to_idx

## 3. Load Train + Test Separately

In [12]:
train_paths, train_labels, class_to_idx = load_data(TRAIN_DIR)
test_paths, test_labels, _ = load_data(TEST_DIR)

print("Train size:", len(train_paths))
print("Test size:", len(test_paths))
print("Classes:", class_to_idx)

Train size: 251
Test size: 66
Classes: {'Covid': 0, 'Normal': 1, 'Viral Pneumonia': 2}


## 4. Distribution Check

In [13]:
print("\nTrain distribution:", Counter(train_labels))
print("Test distribution:", Counter(test_labels))


Train distribution: Counter({0: 111, 1: 70, 2: 70})
Test distribution: Counter({0: 26, 1: 20, 2: 20})


### Save Class Mapping

In [15]:
import json

with open("artifacts/class_mapping.json", "w") as f:
    json.dump(class_to_idx, f)

### Save Train/Val Split

In [16]:
import pandas as pd

df = pd.DataFrame({
    "image_path": train_paths + test_paths,
    "label": train_labels + test_labels,
    "split": ["train"] * len(train_paths) + ["test"] * len(test_paths)
})

df.to_csv("artifacts/data_split.csv", index=False)